In [1]:
from google.colab import drive
import zipfile, os, glob

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from google.colab import drive
import zipfile, os, glob

drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/small_hindi_english_transliteration.zip'
extract_path = '/content/data_csv'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

csv_file = glob.glob(extract_path + '/**/*.csv', recursive=True)[0]
print(" CSV Found:", csv_file)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 CSV Found: /content/data_csv/small_hindi_english_transliteration.csv


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [6]:
from torch.nn.utils.rnn import pad_sequence
class TransliterationDataset(Dataset):
    def __init__(self, csv_file):
        self.data = pd.read_csv(csv_file)
         # Source Hindi text
        self.src_texts = self.data['hindi'].astype(str).tolist()
         # Target English text
        self.tgt_texts = self.data['english'].astype(str).tolist()

        # # Build vocabularies for source and target
        self.src_vocab = self.build_vocab(self.src_texts)
        self.tgt_vocab = self.build_vocab(self.tgt_texts)
        # Reverse mapping for decoding
        self.src_idx2char = {i: c for c, i in self.src_vocab.items()}
        self.tgt_idx2char = {i: c for c, i in self.tgt_vocab.items()}

    def build_vocab(self, texts):
        chars = set(''.join(texts))
        vocab = {'<pad>':0, '<sos>':1, '<eos>':2}
        for i, ch in enumerate(sorted(chars), start=3):
            vocab[ch] = i
        return vocab

    def text_to_seq(self, text, vocab):
        return [vocab['<sos>']] + [vocab.get(ch,0) for ch in text] + [vocab['<eos>']]

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        src_seq = self.text_to_seq(self.src_texts[idx], self.src_vocab)
        tgt_seq = self.text_to_seq(self.tgt_texts[idx], self.tgt_vocab)
        return torch.tensor(src_seq, dtype=torch.long), torch.tensor(tgt_seq, dtype=torch.long)

#  Collate Function to Pad Sequences for batch processing
def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    return src_batch, tgt_batch

In [7]:
# # Encoder Reads the input sequence and creates hidden representations
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)
        return outputs, hidden

# Attention: Helps the decoder focus on important parts of the input at each step
class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        self.attn = nn.Linear(enc_hid_dim + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        if hidden.dim() == 3:
            hidden = hidden[-1]
        batch_size = encoder_outputs.size(0)
        src_len = encoder_outputs.size(1)

        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return torch.softmax(attention, dim=1)

# Decoder with Attention
# Predicts target sequence one token at a time
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim, attention):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim + enc_hid_dim, dec_hid_dim, batch_first=True)
        self.fc_out = nn.Linear(enc_hid_dim + dec_hid_dim + emb_dim, output_dim)
        self.attention = attention

    def forward(self, input, hidden, encoder_outputs):
        input = input.unsqueeze(1)
        embedded = self.embedding(input)

        # Attention
        a = self.attention(hidden, encoder_outputs).unsqueeze(1)
        weighted = torch.bmm(a, encoder_outputs)

        # Concatenate embedding + context
        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, hidden = self.rnn(rnn_input, hidden)
         # Final prediction
        output = output.squeeze(1)
        weighted = weighted.squeeze(1)
        embedded = embedded.squeeze(1)
        prediction = self.fc_out(torch.cat((output, weighted, embedded), dim=1))

        return prediction, hidden



In [15]:

# Seq2Seq wrapper combining Encoder and Decoder
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.75):
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        tgt_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)
        encoder_outputs, hidden = self.encoder(src)
        input = tgt[:,0]

        for t in range(1, tgt_len):
            output, hidden = self.decoder(input, hidden, encoder_outputs)
            outputs[:,t] = output
            teacher_force = np.random.rand() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = tgt[:,t] if teacher_force else top1
        return outputs

In [20]:
# Training Loop
# Performs forward, computes loss, backpropagation, and optimization
def train(model, dataloader, optimizer, criterion, tgt_pad_idx, epochs=20):
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for src, tgt in dataloader:
            src, tgt = src.to(device), tgt.to(device)
            optimizer.zero_grad()
            output = model(src, tgt)
            output_dim = output.shape[-1]
            output = output[:,1:].reshape(-1, output_dim)
            tgt = tgt[:,1:].reshape(-1)
            loss = criterion(output, tgt)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        print(f'Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(dataloader):.4f}')


In [21]:

# Transliteration Function:
# Convert Hindi text into numeric IDs using vocab
#  use Encoder to read and understand the full Hindi word
#  Use Decoder to predict English characters one-by-one
#  Stop when <eos> (end token) is predicted
#  Convert numeric output back into readable English letters
#  Return final transliterated word as a string
def transliterate(model, text, src_vocab, tgt_vocab, tgt_idx2char, max_len=20):
    model.eval()
    src_seq = [src_vocab['<sos>']] + [src_vocab.get(ch,0) for ch in text] + [src_vocab['<eos>']]
    src_tensor = torch.tensor(src_seq).unsqueeze(0).to(device)
    encoder_outputs, hidden = model.encoder(src_tensor)
    input = torch.tensor([tgt_vocab['<sos>']]).to(device)
    result = []
    for _ in range(max_len):
        output, hidden = model.decoder(input, hidden, encoder_outputs)
        top1 = output.argmax(1).item()
        if top1 == tgt_vocab['<eos>']:
            break
        result.append(tgt_idx2char[top1])
        input = torch.tensor([top1]).to(device)
    return ''.join(result)

In [22]:
# Dataset and DataLoader Setup
csv_file = "/content/data_csv/small_hindi_english_transliteration.csv"  # Replace with your CSV path
dataset = TransliterationDataset(csv_file)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

#  Model Setup
INPUT_DIM = len(dataset.src_vocab)
OUTPUT_DIM = len(dataset.tgt_vocab)
ENC_EMB_DIM = 32
DEC_EMB_DIM = 32
HID_DIM = 64

attention = Attention(HID_DIM, HID_DIM)
encoder = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM)
decoder = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, HID_DIM, attention)
model = Seq2Seq(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.CrossEntropyLoss(ignore_index=dataset.tgt_vocab['<pad>'])

#  Training the model
train(model, dataloader, optimizer, criterion, dataset.tgt_vocab['<pad>'], epochs=20)

#  Test Transliteration
test_word = "अनुप्रयोग"
output = transliterate(model, test_word, dataset.src_vocab, dataset.tgt_vocab, dataset.tgt_idx2char)
print(f"Hindi: {test_word} -> Transliteration: {output}")

Epoch 1/20, Loss: 3.2713
Epoch 2/20, Loss: 3.1215
Epoch 3/20, Loss: 3.0274
Epoch 4/20, Loss: 2.9182
Epoch 5/20, Loss: 2.8065
Epoch 6/20, Loss: 2.7181
Epoch 7/20, Loss: 2.5969
Epoch 8/20, Loss: 2.5921
Epoch 9/20, Loss: 2.5468
Epoch 10/20, Loss: 2.4722
Epoch 11/20, Loss: 2.5001
Epoch 12/20, Loss: 2.4029
Epoch 13/20, Loss: 2.4004
Epoch 14/20, Loss: 2.3406
Epoch 15/20, Loss: 2.3275
Epoch 16/20, Loss: 2.3073
Epoch 17/20, Loss: 2.2235
Epoch 18/20, Loss: 2.2213
Epoch 19/20, Loss: 2.2066
Epoch 20/20, Loss: 2.2004
Hindi: अनुप्रयोग -> Transliteration: saara
